In [7]:
import numpy as np
import matplotlib.pyplot as plt
import stackview as sv
import os
import cv2
from tqdm import tqdm
import CtScanReader as ct

In [ ]:
import numpy as np
import cv2

def add_bottom_fan_streaks(
    img,
    num_streaks=100,
    intensity=0.05,
    angle_center_deg=0,     
    angle_spread_deg=4,      # ± spread
    center_offset_ratio=0.7,   # how far down the center is (0.5=center, ~1=bottom)
    streak_noise_scale = 0.3,
    x_shift = 0.2
):
    img = img.copy().astype(np.float32)
    h, w = img.shape

    # Shift center toward bottom
    cx = 0
    cy = int(h * center_offset_ratio)

    streak_layer = np.zeros_like(img)
    for i in range(2):
        for _ in range(num_streaks//2):
            if i == 0:
                cx = 0 + x_shift*w
            elif i == 1:
                cx = 512 - x_shift*w
            # angle range (in radians)
            angle_deg = np.random.normal(
                loc=angle_center_deg,
                scale=angle_spread_deg / 3
            )

            angle_deg = np.clip(
                angle_deg,
                angle_center_deg - angle_spread_deg,
                angle_center_deg + angle_spread_deg
            )
            angle = np.deg2rad(angle_deg)

            # Start outside/near edge
            r1 = np.random.uniform(0.8, 1.3) * max(h, w)
            r2 = 0  # toward center

            x1 = int(cx + (-1)**(i) * r1 * np.cos(angle))
            y1 = int(cy + r1 * np.sin(angle))

            x2 = int(cx + r2 * np.cos(angle))
            y2 = int(cy + r2 * np.sin(angle))

            thickness = 1

            cv2.line(streak_layer, (x1, y1), (x2, y2), 1, thickness)


    streak_layer = cv2.GaussianBlur(streak_layer, (3, 3), 0)

    mask = streak_layer > 0

    noise = np.random.normal(0, streak_noise_scale, size=streak_layer.shape)
    streak_layer[mask] += noise[mask]

    streak_layer = np.clip(streak_layer, 0, None)
    img_artifact = img + -1 * intensity * streak_layer

    return np.clip(img_artifact, 0, 1)

In [9]:
def add_streaks_to_volume(volume,
                          intensity=0.05,
                          angle_center_deg=0,
                          angle_spread_deg = 6,
                          center_offset_ratio=0.8, 
                          streak_noise_scale=0.3):
    
    new_slices = []
    num_slices = volume.shape[0]
    for i in range(num_slices):
        new_img = add_bottom_fan_streaks(volume[i,:,:],
                                        num_streaks = max(int(80-(i*2)), 0),
                                        intensity=intensity,
                                        angle_center_deg=angle_center_deg,
                                        angle_spread_deg = angle_spread_deg,
                                        center_offset_ratio=center_offset_ratio, 
                                        streak_noise_scale=streak_noise_scale)
        new_slices.append(new_img)
    new_volume = np.stack(new_slices, axis=0)
    return new_volume


In [10]:
scan = 'L008'
volume = ct.read_scan(scan, 'raw')
streaked = add_streaks_to_volume(volume,
                                 intensity = 0.07,
                                 angle_center_deg=0,
                                 angle_spread_deg=8,
                                 center_offset_ratio=0.7,
                                 streak_noise_scale = 0.3)

combo = ct.create_side_by_side(volume, streaked)
print(scan)
sv.slice(combo, slice_number=0, continuous_update=1)

L008
